In [3]:
from google.colab import drive
drive.mount('/content/drive')
caminho_arquivo = '/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/transacoes.csv'
print(caminho_arquivo)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/transacoes.csv


In [5]:

import csv

with open(caminho_arquivo, 'r', encoding='utf-8', newline='') as arquivo:
    leitor = csv.DictReader(arquivo)

    for transacao in leitor:
        print(transacao)
        break


{'id': '1', 'data': '2026-01-05', 'cliente_id': 'CLI001', 'tipo': 'credito', 'valor': '3500.00', 'descricao': 'Salário janeiro', 'categoria': 'salario'}


In [26]:


def validar_transacao(transacao):
    """
    Valida uma transação e converte os campos necessários.

    Retorna um dicionário com os dados limpos ou None
    quando algum campo obrigatório é inválido.
    """

    # Valida o identificador da transação.
    try:
        id_transacao = int(transacao['id'].strip())
    except (ValueError, KeyError, AttributeError):
        return None

    # Valida o identificador do cliente.
    try:
        cliente_id = transacao['cliente_id'].strip()
        if not cliente_id:
            return None
    except (KeyError, AttributeError):
        return None

    # Converte a data e captura erros específicos dessa conversão.
    try:
        data = datetime.strptime(
            transacao['data'].strip(),
            '%Y-%m-%d'
        )
    except (ValueError, KeyError, AttributeError):
        return None

    # Valida o tipo da transação.
    try:
        tipo = transacao['tipo'].strip().lower()
        if tipo not in ('credito', 'debito'):
            return None
    except (KeyError, AttributeError):
        return None

    # Converte o valor e captura erros específicos dessa conversão.
    try:
        valor = float(transacao['valor'].strip())
        if valor <= 0:
            return None
    except (ValueError, KeyError, AttributeError):
        return None

    # Monta uma cópia com os campos convertidos.
    transacao_limpa = transacao.copy()
    transacao_limpa['id'] = id_transacao
    transacao_limpa['cliente_id'] = cliente_id
    transacao_limpa['data'] = data
    transacao_limpa['tipo'] = tipo
    transacao_limpa['valor'] = valor

    return transacao_limpa



In [27]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 3: LEITURA DO CSV E LIMPEZA DOS REGISTROS
# ============================================================
#
# OBJETIVO:
# Ler todas as transações do arquivo CSV, validar cada registro
# utilizando a função validar_transacao() e separar os dados
# válidos dos inválidos.
#
# RESPONSABILIDADES:
# 1. Abrir o arquivo CSV localizado no Google Drive.
# 2. Utilizar csv.DictReader para acessar as colunas pelo nome.
# 3. Contabilizar todas as linhas de dados lidas.
# 4. Validar cada transação individualmente.
# 5. Armazenar os registros válidos em uma lista.
# 6. Contabilizar os registros inválidos sem interromper a leitura.
# 7. Exibir um resumo da limpeza no terminal.
#
# TRATAMENTO DE ERROS:
# Se o arquivo não for encontrado, a função captura
# FileNotFoundError e informa o problema.
#
# DEPENDÊNCIA:
# Esta função utiliza validar_transacao(), desenvolvida
# na etapa anterior. Portanto, execute as células anteriores
# antes de executar esta célula.
# ============================================================

import csv


def ler_transacoes(caminho_arquivo):
    """
    Lê o CSV e separa as transações válidas das inválidas.

    Parâmetro:
        caminho_arquivo (str): caminho completo do arquivo CSV.

    Retorno:
        tuple: lista de transações válidas e quantidade de
        transações inválidas.
    """

    # Lista que armazenará somente as transações aprovadas
    # pela função de validação.
    transacoes_validas = []

    # Contadores utilizados para acompanhar a limpeza dos dados.
    total_linhas = 0
    total_invalidas = 0

    try:
        # Abre o arquivo para leitura.
        # O with garante o fechamento do arquivo ao terminar.
        with open(
            caminho_arquivo,
            'r',
            encoding='utf-8',
            newline=''
        ) as arquivo:

            # Interpreta a primeira linha como cabeçalho.
            # Cada registro será retornado como um dicionário.
            leitor = csv.DictReader(arquivo)

            # Percorre todas as linhas de dados do arquivo.
            for transacao in leitor:

                # Conta a linha antes de realizar a validação.
                total_linhas += 1

                # Reutiliza a função criada na etapa anterior.
                transacao_limpa = validar_transacao(transacao)

                # Se o resultado não for None, a transação é válida.
                if transacao_limpa is not None:
                    transacoes_validas.append(transacao_limpa)

                # Caso contrário, incrementa o contador de inválidas.
                else:
                    total_invalidas += 1

        # Exibe o resumo da leitura e da validação.
        print("=" * 45)
        print("RESUMO DA LIMPEZA DOS DADOS")
        print("=" * 45)
        print(f"Total de linhas lidas: {total_linhas}")
        print(f"Linhas válidas:        {len(transacoes_validas)}")
        print(f"Linhas inválidas:      {total_invalidas}")
        print("=" * 45)

        # Retorna os registros válidos e a quantidade de inválidos.
        return transacoes_validas, total_invalidas

    except FileNotFoundError:
        # Informa o problema sem exibir um traceback desnecessário.
        print(f"ERRO: arquivo não encontrado: {caminho_arquivo}")

        # Retorna listas e contadores vazios para sinalizar a falha.
        return [], 0


In [13]:

# Define o caminho do arquivo no Google Drive.
caminho_arquivo = (
    '/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/transacoes.csv'
)

# Lê o CSV e realiza a validação de todas as transações.
transacoes_validas, total_invalidas = ler_transacoes(
    caminho_arquivo
)


RESUMO DA LIMPEZA DOS DADOS
Total de linhas lidas: 25
Linhas válidas:        18
Linhas inválidas:      7


In [14]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 4: GERAÇÃO DO RELATÓRIO FINANCEIRO MENSAL
# ============================================================
#
# OBJETIVO:
# Processar as transações validadas e calcular indicadores
# financeiros agrupados por mês.
#
# MÉTRICAS POR MÊS:
# - Quantidade de transações.
# - Total de créditos e débitos.
# - Saldo mensal: créditos menos débitos.
# - Valor médio das transações.
# - Transação de maior valor.
# - Transação de menor valor.
#
# INFORMAÇÕES GERAIS:
# - Data da transação mais antiga.
# - Data da transação mais recente.
# - Quantidade de dias entre essas duas datas.
#
# A função não altera a lista original de transações.
# Ela retorna um dicionário que será utilizado na exibição
# do relatório e na exportação para JSON.
# ============================================================

def gerar_relatorio(transacoes_validas):
    """
    Agrupa as transações por mês e calcula as métricas financeiras.

    Parâmetro:
        transacoes_validas (list): lista de transações já validadas.

    Retorno:
        dict: período analisado e resumo financeiro mensal.
    """

    # Dicionário que armazenará as métricas de cada mês.
    resumo_mensal = {}

    # Se não houver transações válidas, não há métricas a calcular.
    if not transacoes_validas:
        return {
            'data_mais_antiga': None,
            'data_mais_recente': None,
            'dias_analisados': 0,
            'resumo_mensal': {}
        }

    # Extrai as datas de todas as transações válidas.
    datas = [transacao['data'] for transacao in transacoes_validas]

    # Identifica a primeira e a última data do período analisado.
    data_mais_antiga = min(datas)
    data_mais_recente = max(datas)

    # Calcula a diferença em dias entre as datas.
    dias_analisados = (
        data_mais_recente - data_mais_antiga
    ).days

    # Percorre cada transação para construir o resumo mensal.
    for transacao in transacoes_validas:

        # Converte a data para uma chave no formato AAAA-MM.
        mes = transacao['data'].strftime('%Y-%m')

        # Cria a estrutura inicial do mês na primeira ocorrência.
        if mes not in resumo_mensal:
            resumo_mensal[mes] = {
                'quantidade': 0,
                'total_credito': 0.0,
                'total_debito': 0.0,
                'saldo': 0.0,
                'media': 0.0,
                'maior_valor': 0.0,
                'menor_valor': 0.0
            }

        # Recupera os dados do mês atual.
        resumo = resumo_mensal[mes]

        # Incrementa a quantidade de transações do mês.
        resumo['quantidade'] += 1

        # Obtém o valor financeiro da transação.
        valor = transacao['valor']

        # Soma o valor na categoria financeira correspondente.
        if transacao['tipo'] == 'credito':
            resumo['total_credito'] += valor
        else:
            resumo['total_debito'] += valor

        # Atualiza o maior valor registrado no mês.
        if resumo['quantidade'] == 1:
            resumo['maior_valor'] = valor
            resumo['menor_valor'] = valor
        else:
            resumo['maior_valor'] = max(
                resumo['maior_valor'], valor
            )

            # Atualiza o menor valor registrado no mês.
            resumo['menor_valor'] = min(
                resumo['menor_valor'], valor
            )

    # Calcula o saldo e a média depois de acumular os valores.
    for resumo in resumo_mensal.values():

        # Saldo positivo indica que os créditos superaram os débitos.
        resumo['saldo'] = (
            resumo['total_credito'] - resumo['total_debito']
        )

        # Média aritmética dos valores das transações do mês.
        resumo['media'] = (
            resumo['total_credito'] + resumo['total_debito']
        ) / resumo['quantidade']

    # Retorna os dados organizados para as próximas etapas.
    return {
        'data_mais_antiga': data_mais_antiga,
        'data_mais_recente': data_mais_recente,
        'dias_analisados': dias_analisados,
        'resumo_mensal': resumo_mensal
    }


In [16]:

# Gera as métricas mensais a partir dos registros validados.
relatorio = gerar_relatorio(transacoes_validas)

# Exibe as informações gerais do período analisado.
print("Data mais antiga:", relatorio['data_mais_antiga'])
print("Data mais recente:", relatorio['data_mais_recente'])
print("Dias entre as datas:", relatorio['dias_analisados'])

# Exibe o resumo financeiro de cada mês.
for mes, metricas in relatorio['resumo_mensal'].items():
    print(f"\nMês: {mes}")
    print("Quantidade:", metricas['quantidade'])
    print("Total de crédito:", metricas['total_credito'])
    print("Total de débito:", metricas['total_debito'])
    print("Saldo:", metricas['saldo'])
    print("Média:", metricas['media'])
    print("Maior valor:", metricas['maior_valor'])
    print("Menor valor:", metricas['menor_valor'])


Data mais antiga: 2026-01-05 00:00:00
Data mais recente: 2026-03-27 00:00:00
Dias entre as datas: 81

Mês: 2026-01
Quantidade: 6
Total de crédito: 16000.0
Total de débito: 1796.4
Saldo: 14203.6
Média: 2966.066666666667
Maior valor: 12500.0
Menor valor: 95.9

Mês: 2026-02
Quantidade: 6
Total de crédito: 18500.0
Total de débito: 1036.05
Saldo: 17463.95
Média: 3256.008333333333
Maior valor: 15000.0
Menor valor: 99.9

Mês: 2026-03
Quantidade: 6
Total de crédito: 5700.0
Total de débito: 1145.5
Saldo: 4554.5
Média: 1140.9166666666667
Maior valor: 3500.0
Menor valor: 89.9


In [17]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 5: IDENTIFICAÇÃO DE TRANSAÇÕES SUSPEITAS
# ============================================================
#
# OBJETIVO:
# Identificar transações cujo valor ultrapasse o limite
# estabelecido pelo desafio.
#
# REGRA DE NEGÓCIO:
# Uma transação será sinalizada quando seu valor for
# estritamente superior a R$ 10.000,00.
#
# IMPLEMENTAÇÃO:
# 1. Definir o limite em uma constante.
# 2. Percorrer as transações previamente validadas.
# 3. Comparar o valor de cada transação com o limite.
# 4. Armazenar as transações que ultrapassarem o limite.
#
# IMPORTANTE:
# A sinalização indica apenas que a transação merece atenção.
# Ela não comprova fraude ou irregularidade.
# ============================================================

# Constante que define o valor máximo antes da sinalização.
LIMITE_SUSPEITO = 10000.00


def identificar_suspeitas(transacoes_validas):
    """
    Retorna as transações com valor acima do limite definido.

    Parâmetro:
        transacoes_validas (list): transações já validadas.

    Retorno:
        list: transações que ultrapassam o limite suspeito.
    """

    # Lista que armazenará as transações sinalizadas.
    transacoes_suspeitas = []

    # Analisa cada transação válida individualmente.
    for transacao in transacoes_validas:

        # O critério é estritamente maior que o limite.
        if transacao['valor'] > LIMITE_SUSPEITO:

            # Adiciona a transação completa à lista de suspeitas.
            transacoes_suspeitas.append(transacao)

    # Retorna a lista para utilização no relatório final.
    return transacoes_suspeitas


In [18]:

# Identifica as transações que ultrapassam R$ 10.000,00.
transacoes_suspeitas = identificar_suspeitas(transacoes_validas)

# Exibe os resultados encontrados.
print("=" * 50)
print("TRANSAÇÕES SUSPEITAS")
print("=" * 50)

if transacoes_suspeitas:
    for transacao in transacoes_suspeitas:
        print(
            f"ID: {transacao['id']} | "
            f"Cliente: {transacao['cliente_id']} | "
            f"Data: {transacao['data'].strftime('%Y-%m-%d')} | "
            f"Valor: R$ {transacao['valor']:,.2f}"
            .replace(",", "X").replace(".", ",").replace("X", ".")
        )
else:
    print("Nenhuma transação suspeita encontrada.")


TRANSAÇÕES SUSPEITAS
ID: 5 | Cliente: CLI004 | Data: 2026-01-20 | Valor: R$ 12.500,00
ID: 10 | Cliente: CLI005 | Data: 2026-02-14 | Valor: R$ 15.000,00


In [19]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 6: EXPORTAÇÃO DO RELATÓRIO PARA JSON
# ============================================================
#
# OBJETIVO:
# Exportar os resultados da análise financeira para um arquivo
# JSON, permitindo consultar os indicadores posteriormente.
#
# INFORMAÇÕES EXPORTADAS:
# 1. Data de geração do relatório.
# 2. Quantidade de transações válidas e inválidas.
# 3. Período analisado e quantidade de dias.
# 4. Resumo financeiro de cada mês.
# 5. Lista de transações suspeitas.
#
# DETALHES TÉCNICOS:
# - O módulo json será utilizado para gerar o arquivo.
# - Objetos datetime serão convertidos para texto ISO.
# - ensure_ascii=False preserva os caracteres acentuados.
# - indent=2 facilita a leitura do arquivo gerado.
#
# O arquivo será salvo no Google Drive, na pasta do projeto.
# ============================================================

import json
from datetime import date


def salvar_json(
    caminho_saida,
    relatorio,
    transacoes_validas,
    total_invalidas,
    transacoes_suspeitas
):
    """
    Salva os resultados da análise financeira em formato JSON.

    Parâmetros:
        caminho_saida (str): caminho do arquivo JSON.
        relatorio (dict): métricas e período analisado.
        transacoes_validas (list): transações aprovadas.
        total_invalidas (int): quantidade de registros inválidos.
        transacoes_suspeitas (list): transações sinalizadas.

    Retorno:
        bool: True se o arquivo for salvo; False em caso de erro.
    """

    # Monta a estrutura principal do relatório.
    dados_json = {
        # Registra a data em que o relatório foi gerado.
        'gerado_em': date.today().isoformat(),

        # Registra as quantidades de transações processadas.
        'total_transacoes_validas': len(transacoes_validas),
        'total_transacoes_invalidas': total_invalidas,

        # Armazena as informações gerais do período analisado.
        'periodo_analisado': {
            'data_mais_antiga': (
                relatorio['data_mais_antiga'].strftime('%Y-%m-%d')
                if relatorio['data_mais_antiga'] else None
            ),
            'data_mais_recente': (
                relatorio['data_mais_recente'].strftime('%Y-%m-%d')
                if relatorio['data_mais_recente'] else None
            ),
            'dias_analisados': relatorio['dias_analisados']
        },

        # Inclui as métricas financeiras agrupadas por mês.
        'resumo_mensal': relatorio['resumo_mensal'],

        # Inclui os registros que ultrapassaram o limite suspeito.
        'transacoes_suspeitas': [
            {
                'id': transacao['id'],
                'cliente_id': transacao['cliente_id'],
                'data': transacao['data'].strftime('%Y-%m-%d'),
                'valor': transacao['valor']
            }
            for transacao in transacoes_suspeitas
        ]
    }

    try:
        # Abre ou cria o arquivo JSON para escrita.
        with open(
            caminho_saida,
            'w',
            encoding='utf-8'
        ) as arquivo:

            # Serializa o dicionário para o formato JSON.
            json.dump(
                dados_json,
                arquivo,
                ensure_ascii=False,
                indent=2
            )

        # Confirma o caminho utilizado na exportação.
        print(f"Relatório JSON salvo em: {caminho_saida}")
        return True

    except (OSError, TypeError, ValueError) as erro:
        # Informa problemas de escrita ou serialização.
        print(f"Erro ao salvar o relatório JSON: {erro}")
        return False


In [21]:

# Define o destino do relatório na pasta do projeto.
caminho_json = (
    '/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/relatorio.json'
)

# Exporta os resultados calculados nas etapas anteriores.
arquivo_salvo = salvar_json(
    caminho_saida=caminho_json,
    relatorio=relatorio,
    transacoes_validas=transacoes_validas,
    total_invalidas=total_invalidas,
    transacoes_suspeitas=transacoes_suspeitas
)

# Informa o resultado da operação.
if arquivo_salvo:
    print("Exportação concluída com sucesso.")
else:
    print("Não foi possível exportar o relatório.")


Relatório JSON salvo em: /content/drive/MyDrive/Estudo/Rocketseat/Desafio3/relatorio.json
Exportação concluída com sucesso.


In [22]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 7: EXIBIÇÃO FORMATADA DO RELATÓRIO
# ============================================================
#
# OBJETIVO:
# Exibir os resultados da análise financeira de maneira clara,
# organizada e adequada para leitura humana.
#
# INFORMAÇÕES EXIBIDAS:
# - Quantidade de registros válidos e inválidos.
# - Período analisado e quantidade de dias.
# - Indicadores financeiros de cada mês.
# - Transações que ultrapassaram o limite suspeito.
#
# FORMATAÇÃO:
# - Valores monetários com duas casas decimais.
# - Prefixo R$ e separadores no padrão brasileiro.
# - Separadores visuais entre as seções do relatório.
#
# DEPENDÊNCIAS:
# Esta função utiliza os resultados calculados anteriormente.
# Ela não recalcula as métricas nem modifica os dados.
# ============================================================


def formatar_moeda(valor):
    """
    Formata um número no padrão monetário brasileiro.

    Exemplo:
        1234.50 -> R$ 1.234,50
    """

    # Aplica separadores de milhar e casas decimais.
    valor_formatado = f"{valor:,.2f}"

    # Inverte os separadores para o padrão brasileiro.
    valor_formatado = (
        valor_formatado
        .replace(",", "X")
        .replace(".", ",")
        .replace("X", ".")
    )

    return f"R$ {valor_formatado}"


def exibir_relatorio(
    relatorio,
    transacoes_validas,
    total_invalidas,
    transacoes_suspeitas
):
    """
    Exibe o relatório financeiro completo no terminal.

    Parâmetros:
        relatorio (dict): métricas mensais e período analisado.
        transacoes_validas (list): registros aprovados.
        total_invalidas (int): quantidade de registros rejeitados.
        transacoes_suspeitas (list): registros sinalizados.
    """

    # --------------------------------------------------------
    # 1. RESUMO DA LIMPEZA DOS DADOS
    # --------------------------------------------------------

    print("=" * 55)
    print("RESUMO DA LIMPEZA DOS DADOS")
    print("=" * 55)

    print(f"Total de linhas lidas: {len(transacoes_validas) + total_invalidas}")
    print(f"Linhas válidas:        {len(transacoes_validas)}")
    print(f"Linhas inválidas:      {total_invalidas}")

    # --------------------------------------------------------
    # 2. PERÍODO ANALISADO
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("PERÍODO ANALISADO")
    print("=" * 55)

    data_inicial = relatorio['data_mais_antiga']
    data_final = relatorio['data_mais_recente']

    if data_inicial and data_final:
        print(
            f"Período: {data_inicial:%Y-%m-%d} "
            f"-> {data_final:%Y-%m-%d}"
        )
        print(f"Dias entre as datas: {relatorio['dias_analisados']}")
    else:
        print("Não existem transações válidas para analisar.")

    # --------------------------------------------------------
    # 3. MÉTRICAS FINANCEIRAS MENSAIS
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("RELATÓRIO FINANCEIRO MENSAL")
    print("=" * 55)

    # Ordena os meses cronologicamente.
    resumo_mensal = relatorio['resumo_mensal']

    for mes in sorted(resumo_mensal):
        metricas = resumo_mensal[mes]

        print(f"\nMês: {mes}")
        print(f"  Transações:    {metricas['quantidade']}")
        print(f"  Total crédito: {formatar_moeda(metricas['total_credito'])}")
        print(f"  Total débito:  {formatar_moeda(metricas['total_debito'])}")
        print(f"  Saldo:         {formatar_moeda(metricas['saldo'])}")
        print(f"  Média:         {formatar_moeda(metricas['media'])}")
        print(f"  Maior valor:   {formatar_moeda(metricas['maior_valor'])}")
        print(f"  Menor valor:   {formatar_moeda(metricas['menor_valor'])}")

    # --------------------------------------------------------
    # 4. TRANSAÇÕES SUSPEITAS
    # --------------------------------------------------------

    print("\n" + "=" * 55)
    print("TRANSAÇÕES SUSPEITAS")
    print("=" * 55)

    if transacoes_suspeitas:
        for transacao in transacoes_suspeitas:
            data_formatada = transacao['data'].strftime('%Y-%m-%d')

            print(
                f"ID: {transacao['id']} | "
                f"Cliente: {transacao['cliente_id']} | "
                f"Data: {data_formatada} | "
                f"Valor: {formatar_moeda(transacao['valor'])}"
            )
    else:
        print("Nenhuma transação suspeita encontrada.")

    print("\n" + "=" * 55)
    print("FIM DO RELATÓRIO")
    print("=" * 55)


In [23]:

exibir_relatorio(
    relatorio=relatorio,
    transacoes_validas=transacoes_validas,
    total_invalidas=total_invalidas,
    transacoes_suspeitas=transacoes_suspeitas
)


RESUMO DA LIMPEZA DOS DADOS
Total de linhas lidas: 25
Linhas válidas:        18
Linhas inválidas:      7

PERÍODO ANALISADO
Período: 2026-01-05 -> 2026-03-27
Dias entre as datas: 81

RELATÓRIO FINANCEIRO MENSAL

Mês: 2026-01
  Transações:    6
  Total crédito: R$ 16.000,00
  Total débito:  R$ 1.796,40
  Saldo:         R$ 14.203,60
  Média:         R$ 2.966,07
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 95,90

Mês: 2026-02
  Transações:    6
  Total crédito: R$ 18.500,00
  Total débito:  R$ 1.036,05
  Saldo:         R$ 17.463,95
  Média:         R$ 3.256,01
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações:    6
  Total crédito: R$ 5.700,00
  Total débito:  R$ 1.145,50
  Saldo:         R$ 4.554,50
  Média:         R$ 1.140,92
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

TRANSAÇÕES SUSPEITAS
ID: 5 | Cliente: CLI004 | Data: 2026-01-20 | Valor: R$ 12.500,00
ID: 10 | Cliente: CLI005 | Data: 2026-02-14 | Valor: R$ 15.000,00

FIM DO RELATÓRIO


In [28]:

# ============================================================
# PROJETO: ANÁLISE FINANCEIRA COM PYTHON - CLEARBANK
# ETAPA 8: EXECUÇÃO PRINCIPAL DO PROJETO
# ============================================================
#
# OBJETIVO:
# Coordenar a execução de todas as etapas da análise financeira,
# desde a leitura do CSV até a geração do relatório JSON.
#
# FLUXO DE EXECUÇÃO:
# 1. Verificar se os arquivos de entrada e saída estão acessíveis.
# 2. Ler o CSV e validar os registros.
# 3. Gerar o relatório financeiro mensal.
# 4. Identificar as transações suspeitas.
# 5. Exibir o relatório formatado no terminal.
# 6. Exportar os resultados para relatorio.json.
#
# IMPORTANTE:
# Execute as células anteriores antes desta célula, pois ela
# utiliza as funções e variáveis definidas nas etapas anteriores.
# ============================================================

import os

# Define os caminhos dos arquivos utilizados no projeto.
caminho_arquivo = (
    '/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/transacoes.csv'
)

caminho_json = (
    '/content/drive/MyDrive/Estudo/Rocketseat/Desafio3/relatorio.json'
)

# Verifica se o CSV existe antes de iniciar o processamento.
if not os.path.isfile(caminho_arquivo):
    print(f"ERRO: arquivo CSV não encontrado: {caminho_arquivo}")

else:
    # ETAPA 1: Ler o CSV e separar registros válidos e inválidos.
    transacoes_validas, total_invalidas = ler_transacoes(
        caminho_arquivo
    )

    # ETAPA 2: Calcular as métricas financeiras mensais.
    relatorio = gerar_relatorio(transacoes_validas)

    # ETAPA 3: Identificar transações acima do limite definido.
    transacoes_suspeitas = identificar_suspeitas(
        transacoes_validas
    )

    # ETAPA 4: Exibir os resultados no terminal.
    exibir_relatorio(
        relatorio=relatorio,
        transacoes_validas=transacoes_validas,
        total_invalidas=total_invalidas,
        transacoes_suspeitas=transacoes_suspeitas
    )

    # ETAPA 5: Exportar os resultados para o arquivo JSON.
    arquivo_salvo = salvar_json(
        caminho_saida=caminho_json,
        relatorio=relatorio,
        transacoes_validas=transacoes_validas,
        total_invalidas=total_invalidas,
        transacoes_suspeitas=transacoes_suspeitas
    )

    # Apresenta o resultado final da execução.
    if arquivo_salvo:
        print("\nProcessamento concluído.")
        print("Relatório JSON gerado com sucesso.")
    else:
        print("\nProcessamento concluído com falha na exportação JSON.")


RESUMO DA LIMPEZA DOS DADOS
Total de linhas lidas: 25
Linhas válidas:        18
Linhas inválidas:      7
RESUMO DA LIMPEZA DOS DADOS
Total de linhas lidas: 25
Linhas válidas:        18
Linhas inválidas:      7

PERÍODO ANALISADO
Período: 2026-01-05 -> 2026-03-27
Dias entre as datas: 81

RELATÓRIO FINANCEIRO MENSAL

Mês: 2026-01
  Transações:    6
  Total crédito: R$ 16.000,00
  Total débito:  R$ 1.796,40
  Saldo:         R$ 14.203,60
  Média:         R$ 2.966,07
  Maior valor:   R$ 12.500,00
  Menor valor:   R$ 95,90

Mês: 2026-02
  Transações:    6
  Total crédito: R$ 18.500,00
  Total débito:  R$ 1.036,05
  Saldo:         R$ 17.463,95
  Média:         R$ 3.256,01
  Maior valor:   R$ 15.000,00
  Menor valor:   R$ 99,90

Mês: 2026-03
  Transações:    6
  Total crédito: R$ 5.700,00
  Total débito:  R$ 1.145,50
  Saldo:         R$ 4.554,50
  Média:         R$ 1.140,92
  Maior valor:   R$ 3.500,00
  Menor valor:   R$ 89,90

TRANSAÇÕES SUSPEITAS
ID: 5 | Cliente: CLI004 | Data: 2026-01-20 |